安装opencompass：Kaggle上已经为我们准备好了其他常用包，只需安装opencompass用于评测即可。如果不在Kaggle上运行，则还需要安装其他必要包。

In [1]:
# !pip install "opencompass[full]"
# !pip install pytorch transformers datasets "opencompass[full]"

# 指令微调

In [2]:
"""
The main program for finetuning LLMs with Huggingface Transformers Library.

ALL SECTIONS WHERE CODE POSSIBLY NEEDS TO BE FILLED IN ARE MARKED AS TODO.
"""

import argparse
from dataclasses import dataclass, field
from typing import Optional, List, Dict
import sys
import torch
from transformers import TrainingArguments, HfArgumentParser, Trainer, AutoTokenizer, AutoModelForCausalLM
import datasets
from datasets import load_dataset

/opt/conda/envs/opencompass/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Define the arguments required for the main program.
# NOTE: You can customize any arguments you need to pass in.
@dataclass
class ModelArguments:
    """Arguments for model
    """
    model_name_or_path: Optional[str] = field(
        default="/workspace/qwen",
        metadata={
            "help": "The path to the LLM to fine-tune or its name on the Hugging Face Hub."
        }
    )
    torch_dtype: Optional[str] = field(
        default=None,
        metadata={
            "help": (
                "Override the default `torch.dtype` and load the model under this dtype."
            ),
            "choices": ["bfloat16", "float16", "float32"],
        },
    )
    # TODO: add your model arguments here
    device: Optional[str] = field(
        default=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        metadata={
            "help": (
                "Override the default `torch.dtype` and load the model under this dtype."
            ),
            "choices": ["bfloat16", "float16", "float32"],
        },
    )
    pass


@dataclass
class DataArguments:
    """Arguments for data
    """
    dataset_path: Optional[str] = field(
        default=None,
        metadata={
            "help": "The path to the fine-tuning dataset or its name on the Hugging Face Hub."
        }
    )
    # TODO: add your data arguments here

In [4]:
# The main function
# NOTE You can customize some logs to monitor your program.
def finetune():
    # TODO Step 1: Define an arguments parser and parse the arguments
    # NOTE Three parts: model arguments, data arguments, and training arguments
    # HINT: Refer to 
    #   * https://huggingface.co/docs/transformers/v4.46.3/en/internal/trainer_utils#transformers.HfArgumentParser
    #   * https://huggingface.co/docs/transformers/v4.46.3/en/main_classes/trainer#transformers.TrainingArguments
    parser = HfArgumentParser((ModelArguments, DataArguments, TrainingArguments))
    model_args, data_args, training_args = parser.parse_args_into_dataclasses()
    print(model_args.device)

    # TODO Step 2: Load tokenizer and model2
    # HINT 1: Refer to
    #   * https://huggingface.co/docs/transformers/v4.46.3/en/main_classes/tokenizer#tokenizer
    #   * https://huggingface.co/docs/transformers/v4.46.3/en/model_doc/qwen2
    # HINT 2: To save training GPU memory, you need to set the model's parameter precision to half-precision (float16 or bfloat16).
    #         You may also check other strategies to save the memory!
    #   * https://huggingface.co/docs/transformers/v4.46.3/en/model_doc/llama2#usage-tips
    #   * https://huggingface.co/docs/transformers/perf_train_gpu_one
    #   * https://www.53ai.com/news/qianyanjishu/2024052494875.html
    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(model_args.model_name_or_path).to(model_args.device)


    # TODO Step 3: Load dataset
    # HINT: https://huggingface.co/docs/datasets/v3.1.0/en/package_reference/main_classes#datasets.Dataset
    dataset = load_dataset("workspace/alpaca-cleaned")
    # print(len(dataset["train"]))

    # TODO Step 4: Define the data collator function
    # NOTE During training, for each model parameter update, we fetch a batch of data, perform a forward and backward pass,
    # and then update the model parameters. The role of the data collator is to process the data (e.g., padding the data within
    # a batch to the same length) and format the batch into the input required by the model.
    #
    # In this assignment, the purpose of the custom data_collator is to process each batch of data from the dataset loaded in
    # Step 3 into the format required by the model. This includes tasks such as tokenizing the data, converting each token into 
    # an ID sequence, applying padding, and preparing labels.
    # 
    # HINT:
    #   * Before implementation, you should:
    #      1. Clearly understand the format of each sample in the dataset loaded in Step 3.
    #      2. Understand the input format required by the model (https://huggingface.co/docs/transformers/model_doc/qwen2#transformers.Qwen2ForCausalLM).
    #         Reading its source code also helps!

    def data_collator(batch: List[Dict]):
        """
        batch: list of dict, each dict of the list is a sample in the dataset.
        """
        inputs = []
        targets = []
        # print("im data_collator ", len(batch))

        for sample in batch:
            instruction = sample.get("instruction", "")
            input_text = sample.get("input", "")
            target_text = sample.get("output", "")
            if input_text:
                combined_input = f"{instruction}\n{input_text}"
            else:
                combined_input = instruction
            inputs.append(combined_input)
            targets.append(target_text)

        model_inputs = tokenizer(
            inputs,
            max_length=1024,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        
        labels = tokenizer(
            targets,
            max_length=1024,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )["input_ids"]
        
        if model_inputs["input_ids"].shape[1] != labels.shape[1]:
            max_len = max(model_inputs["input_ids"].shape[1], labels.shape[1])
            model_inputs["input_ids"] = torch.nn.functional.pad(model_inputs["input_ids"], (0, max_len - model_inputs["input_ids"].shape[1]),value=0)
            model_inputs["attention_mask"] = torch.nn.functional.pad(model_inputs["attention_mask"], (0, max_len - model_inputs["attention_mask"].shape[1]),value=0)
            labels = torch.nn.functional.pad(labels, (0, max_len - labels.shape[1]), value=-100)

        return {
            "input_ids": model_inputs["input_ids"],
            "attention_mask": model_inputs["attention_mask"],
            "labels": labels,
        }
    
    dataset["train"] = dataset["train"].select(range(26000))
    # TODO Step 5: Define the Trainer
    # HINT: https://huggingface.co/docs/transformers/main_classes/trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"], 
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    # Step 6: Train!
    trainer.train()

In [ ]:
# Pass your training arguments.
# NOTE [IMPORTANT!!!] DO NOT FORGET TO PASS PROPER ARGUMENTS TO SAVE YOUR CHECKPOINTS!!!
sys.argv = [
    "notebook", 
    "--output_dir", "/workspace/results",            # 模型保存路径
    "--num_train_epochs", "1",             # 训练轮数
    "--learning_rate", "2e-5",             # 学习率
    "--per_device_train_batch_size", "1", # 每个设备的批量大小
    "--fp16",                              # 开启 AMP 混合精度训练
    "--logging_dir", "/workspace/logs",     # 日志保存路径
    "--logging_strategy", "steps",         # 日志记录策略
    "--logging_steps", "10000",               # 每 100000 步记录一次日志
    "--save_strategy", "epoch",            # 每个 epoch 保存一次
    "--report_to", "none",                  # 禁止自动将日志发送到 WandB 等服务22
    "--remove_unused_columns", "False"
]
finetune()

cuda


/tmp/ipykernel_63261/145970032.py:96: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


# 评测模型

In [6]:
PLM_MODEL_PATH = "/workspace/qwen"
SFT_MODEL_PATH = "/workspace/results/checkpoint-26000"

如果你有多个GPU，可以修改下面的--hf-num-gpus参数来加速评测。

In [ ]:
# !opencompass \
#     --datasets mmlu_ppl hellaswag_clean_ppl winogrande_ll ARC_e_ppl ARC_c_clean_ppl SuperGLUE_BoolQ_few_shot_ppl \
#     --summarizer example \
#     --hf-type base \
#     --hf-path {PLM_MODEL_PATH} \
#     --tokenizer-kwargs padding_side="left" truncation="left" \
#     --max-seq-len 2048 \
#     --batch-size 1 \
#     --hf-num-gpus 1 \
#     --work-dir "/workspace/outputs/evals/plm" \
#     --debug

/opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/__init__.py:19: UserWarning: Starting from v0.4.0, all AMOTIC configuration files currently located in `./configs/datasets`, `./configs/models`, and `./configs/summarizers` will be migrated to the `opencompass/configs/` package. Please update your configuration file paths accordingly.
  _warn_about_config_migration()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


12/29 07:28:35 - OpenCompass - INFO - Loading mmlu_ppl: /opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/configs/./datasets/mmlu/mmlu_ppl.py
12/29 07:28:35 - OpenCompass - INFO - Loading hellaswag_clean_ppl: /opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/configs/./datasets/hellaswag/hellaswag_clean_ppl.py
12/29 07:28:35 - OpenCompass - INFO - Loading winogrande_ll: /opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/configs/./datasets/winogrande/winogrande_ll.py
12/29 07:28:35 - OpenCompass - INFO - Loading ARC_e_ppl: /opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/configs/./datasets/ARC_e/ARC_e_ppl.py
12/29 07:28:35 - OpenCompass - INFO - Loading ARC_c_clean_ppl: /opt/conda/envs/opencompass/lib/python3.10/site-packages/opencompass/configs/./datasets/ARC_c/ARC_c_clean_ppl.py
12/29 07:28:35 - OpenCompass - INFO - Loading SuperGLUE_BoolQ_few_shot_ppl: /opt/conda/envs/opencompass/lib/python3.10/site-packages/o

In [ ]:
!opencompass \
    --datasets mmlu_ppl hellaswag_clean_ppl winogrande_ll ARC_e_ppl ARC_c_clean_ppl SuperGLUE_BoolQ_few_shot_ppl \
    --summarizer example \
    --hf-type base \
    --hf-path {SFT_MODEL_PATH} \
    --tokenizer-kwargs padding_side="left" truncation="left" \
    --max-seq-len 1024 \
    --batch-size 1 \
    --hf-num-gpus 1 \
    --work-dir "outputs/evals/sft" \
    --debug